In [2]:
import json
from datetime import datetime
from pathlib import Path


INPUT_JSON = "./raw/presidency/presidency.json"
OUTPUT_JSON = "./processed/presidency_enriched.json"


# Başkanlık seçim tarihleri. Gerekirse genişletilir/düzeltilir.
ELECTION_DATES = {
    1789: "1789-01-07",
    1792: "1792-11-02",
    1796: "1796-11-04",
    1800: "1800-10-31",
    1804: "1804-11-02",
    1808: "1808-11-04",
    1812: "1812-10-30",
    1816: "1816-11-01",
    1820: "1820-11-01",
    1824: "1824-10-26",
    1828: "1828-10-31",
    1832: "1832-11-02",
    1836: "1836-11-03",
    1840: "1840-10-30",
    1844: "1844-11-01",
    1848: "1848-11-07",
    1852: "1852-11-02",
    1856: "1856-11-04",
    1860: "1860-11-06",
    1864: "1864-11-08",
    1868: "1868-11-03",
    1872: "1872-11-05",
    1876: "1876-11-07",
    1880: "1880-11-02",
    1884: "1884-11-04",
    1888: "1888-11-06",
    1892: "1892-11-08",
    1896: "1896-11-03",
    1900: "1900-11-06",
    1904: "1904-11-08",
    1908: "1908-11-03",
    1912: "1912-11-05",
    1916: "1916-11-07",
    1920: "1920-11-02",
    1924: "1924-11-04",
    1928: "1928-11-06",
    1932: "1932-11-08",
    1936: "1936-11-03",
    1940: "1940-11-05",
    1944: "1944-11-07",
    1948: "1948-11-02",
    1952: "1952-11-04",
    1956: "1956-11-06",
    1960: "1960-11-08",
    1964: "1964-11-03",
    1968: "1968-11-05",
    1972: "1972-11-07",
    1976: "1976-11-02",
    1980: "1980-11-04",
    1984: "1984-11-06",
    1988: "1988-11-08",
    1992: "1992-11-03",
    1996: "1996-11-05",
    2000: "2000-11-07",
    2004: "2004-11-02",
    2008: "2008-11-04",
    2012: "2012-11-06",
    2016: "2016-11-08",
    2020: "2020-11-03",
}


# Başkan -> parti.
# Not: bazı başkanların parti aidiyeti tarihsel olarak değişken/karmaşıktır.
PRESIDENT_PARTY = {
    "George Washington": "Independent",
    "John Adams": "Federalist",
    "Thomas Jefferson": "Democratic-Republican",
    "James Madison": "Democratic-Republican",
    "James Monroe": "Democratic-Republican",
    "John Quincy Adams": "Democratic-Republican",
    "Andrew Jackson": "Democratic",
    "Martin Van Buren": "Democratic",
    "William Henry Harrison": "Whig",
    "John Tyler": "Whig",
    "James K. Polk": "Democratic",
    "Zachary Taylor": "Whig",
    "Millard Fillmore": "Whig",
    "Franklin Pierce": "Democratic",
    "James Buchanan": "Democratic",
    "Abraham Lincoln": "Republican",
    "Andrew Johnson": "National Union",
    "Ulysses S. Grant": "Republican",
    "Rutherford B. Hayes": "Republican",
    "James A. Garfield": "Republican",
    "Chester A. Arthur": "Republican",
    "Grover Cleveland": "Democratic",
    "Benjamin Harrison": "Republican",
    "William McKinley": "Republican",
    "Theodore Roosevelt": "Republican",
    "William Howard Taft": "Republican",
    "Woodrow Wilson": "Democratic",
    "Warren G. Harding": "Republican",
    "Calvin Coolidge": "Republican",
    "Herbert Hoover": "Republican",
    "Franklin D. Roosevelt": "Democratic",
    "Harry S. Truman": "Democratic",
    "Dwight D. Eisenhower": "Republican",
    "John F. Kennedy": "Democratic",
    "Lyndon B. Johnson": "Democratic",
    "Richard Nixon": "Republican",
    "Gerald R. Ford": "Republican",
    "Jimmy Carter": "Democratic",
    "Ronald Reagan": "Republican",
    "George Bush": "Republican",
    "Bill Clinton": "Democratic",
    "George W. Bush": "Republican",
    "Barack Obama": "Democratic",
    "Donald Trump": "Republican",
    "Joseph R. Biden": "Democratic",
}


# Seçim kazananları.
ELECTION_WINNERS = {
    1789: "George Washington",
    1792: "George Washington",
    1796: "John Adams",
    1800: "Thomas Jefferson",
    1804: "Thomas Jefferson",
    1808: "James Madison",
    1812: "James Madison",
    1816: "James Monroe",
    1820: "James Monroe",
    1824: "John Quincy Adams",
    1828: "Andrew Jackson",
    1832: "Andrew Jackson",
    1836: "Martin Van Buren",
    1840: "William Henry Harrison",
    1844: "James K. Polk",
    1848: "Zachary Taylor",
    1852: "Franklin Pierce",
    1856: "James Buchanan",
    1860: "Abraham Lincoln",
    1864: "Abraham Lincoln",
    1868: "Ulysses S. Grant",
    1872: "Ulysses S. Grant",
    1876: "Rutherford B. Hayes",
    1880: "James A. Garfield",
    1884: "Grover Cleveland",
    1888: "Benjamin Harrison",
    1892: "Grover Cleveland",
    1896: "William McKinley",
    1900: "William McKinley",
    1904: "Theodore Roosevelt",
    1908: "William Howard Taft",
    1912: "Woodrow Wilson",
    1916: "Woodrow Wilson",
    1920: "Warren G. Harding",
    1924: "Calvin Coolidge",
    1928: "Herbert Hoover",
    1932: "Franklin D. Roosevelt",
    1936: "Franklin D. Roosevelt",
    1940: "Franklin D. Roosevelt",
    1944: "Franklin D. Roosevelt",
    1948: "Harry S. Truman",
    1952: "Dwight D. Eisenhower",
    1956: "Dwight D. Eisenhower",
    1960: "John F. Kennedy",
    1964: "Lyndon B. Johnson",
    1968: "Richard Nixon",
    1972: "Richard Nixon",
    1976: "Jimmy Carter",
    1980: "Ronald Reagan",
    1984: "Ronald Reagan",
    1988: "George Bush",
    1992: "Bill Clinton",
    1996: "Bill Clinton",
    2000: "George W. Bush",
    2004: "George W. Bush",
    2008: "Barack Obama",
    2012: "Barack Obama",
    2016: "Donald Trump",
    2020: "Joseph R. Biden",
}


def parse_date(date_value):
    if date_value is None:
        return None

    # pandas Timestamp / datetime / date destekle
    if hasattr(date_value, "date"):
        return date_value.date()

    date_string = str(date_value).strip()

    if date_string in ["", "None", "nan", "NaT"]:
        return None

    formats = [
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d",
        "%B %d, %Y",
        "%b %d, %Y",
        "%Y"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(date_string, fmt).date()
        except ValueError:
            pass

    return None


def get_next_election(doc_date):
    if doc_date is None:
        return None, None, None

    elections = [
        (year, parse_date(date))
        for year, date in ELECTION_DATES.items()
    ]

    elections = sorted(elections, key=lambda x: x[1])

    for year, election_date in elections:
        if doc_date <= election_date:
            days_before = (election_date - doc_date).days
            return f"US_{year}", election_date.isoformat(), days_before

    return None, None, None


def get_role(speaker, next_election_id):
    if not next_election_id:
        return None

    election_year = int(next_election_id.replace("US_", ""))
    winner = ELECTION_WINNERS.get(election_year)

    if speaker == winner:
        return "winner"

    # Presidency corpus mostly contains presidents, so if the speaker is not the winner
    # but is speaking before a future election, he is usually incumbent or non-candidate president.
    # Candidate-loser/challenger flags require full candidate table.
    return "incumbent_or_non_candidate_president"


with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)


for speaker, sections in data.items():
    party = PRESIDENT_PARTY.get(speaker)

    for section_name, documents in sections.items():
        for doc in documents:
            doc_date = parse_date(doc.get("document_date"))

            next_election, next_election_date, days_before = get_next_election(doc_date)

            doc["speaker"] = speaker
            doc["party"] = party
            doc["document_type"] = section_name

            doc["next_election"] = next_election
            doc["next_election_date"] = next_election_date
            doc["days_before_next_election"] = days_before

            doc["speaker_role_next_election"] = get_role(
                speaker=speaker,
                next_election_id=next_election
            )


with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)


print(f"Saved enriched file: {OUTPUT_JSON}")

Saved enriched file: ./processed/presidency_enriched.json
